# General-Purpose Raster Tile Merger
## TILES MOSAICING SCRIPT

This Python script merges GeoTIFF tiles while intelligently handling nodata values.

**Key Features:**
- Preserves legitimate negative values (elevation, temperature, etc.)
- Automatically detects nodata from tile metadata
- Unifies multiple nodata values into a single value
- Memory-efficient windowed processing for large files
- Parallel processing for faster results

**How to Use:**
1. Specify the directory containing RP subfolders (not the RP subfolder itself)
2. Choose merge mode:
   - **General**: Preserves negative values, detects nodata automatically
   - **Fathom**: Converts negatives to 0, sets nodata=0 (for flood depth data)
3. Output files are created in the parent directory of each subfolder

**Requirements:**
- GDAL library must be installed in your Python environment
- Works best on multi-core CPUs for parallel processing

In [ ]:
import os
import sys
import concurrent.futures
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
import tkinter as tk
from tkinter import filedialog

# Add utility path to access merge_utils_general
utility_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'utility'))
if utility_path not in sys.path:
    sys.path.insert(0, utility_path)

from merge_utils_general import merge_tifs_general, merge_tifs_fathom

def select_folder():
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.geometry("1x1+0+0")
    root.withdraw()
    
    root.after(0, lambda: root.focus_force())
    folder_selected = filedialog.askdirectory(master=root)
    
    root.destroy()
    return folder_selected

class GeneralTilesMosaicingTool:
    def __init__(self):
        self.directory_input = widgets.Text(
            value='',
            placeholder='Enter the directory path',
            description='Directory:',
            disabled=False,
            layout=widgets.Layout(width='500px')
        )
        
        self.search_button = widgets.Button(
            description='Search Folder',
            disabled=False,
            button_style='',
            tooltip='Click to search for a folder',
            layout=widgets.Layout(width='120px')
        )
        
        self.merge_mode = widgets.Dropdown(
            options=[
                ('General (preserve negatives, auto-detect nodata)', 'general'),
                ('Fathom (convert negatives to 0, nodata=0)', 'fathom')
            ],
            value='general',
            description='Merge Mode:',
            disabled=False,
            layout=widgets.Layout(width='500px')
        )
        
        self.target_nodata_input = widgets.Text(
            value='',
            placeholder='Leave empty for auto-detection',
            description='Target NoData:',
            disabled=False,
            layout=widgets.Layout(width='250px')
        )
        
        self.unify_nodata_checkbox = widgets.Checkbox(
            value=True,
            description='Unify multiple nodata values into target nodata',
            disabled=False,
            indent=False,
            layout=widgets.Layout(width='500px')
        )
        
        self.process_button = widgets.Button(
            description='Process Tiles',
            disabled=False,
            button_style='info',
            tooltip='Click to start processing',
            layout=widgets.Layout(width='120px')
        )
        
        self.delete_checkbox = widgets.Checkbox(
            value=False,
            description='Delete original subfolders after processing',
            disabled=False,
            indent=False,
            layout=widgets.Layout(width='500px')
        )
        
        self.output = widgets.Output()
        
        self.search_button.on_click(self.on_search_button_clicked)
        self.process_button.on_click(self.on_process_button_clicked)
        self.merge_mode.observe(self.on_merge_mode_change, names='value')
        
        # Layout
        display(widgets.VBox([
            widgets.HBox([self.directory_input, self.search_button]),
            self.merge_mode,
            widgets.HBox([self.target_nodata_input, self.unify_nodata_checkbox]),
            widgets.HBox([self.process_button, self.delete_checkbox]),
            self.output
        ]))
    
    def on_merge_mode_change(self, change):
        # Disable nodata options for Fathom mode
        if change['new'] == 'fathom':
            self.target_nodata_input.disabled = True
            self.unify_nodata_checkbox.disabled = True
            self.target_nodata_input.value = '0'
        else:
            self.target_nodata_input.disabled = False
            self.unify_nodata_checkbox.disabled = False
    
    def on_search_button_clicked(self, b):
        folder_path = select_folder()
        if folder_path:
            self.directory_input.value = folder_path
    
    def process_subdir(self, subdir):
        """Process a single subdirectory"""
        try:
            if self.merge_mode.value == 'fathom':
                result = merge_tifs_fathom(subdir)
            else:
                # Parse target nodata
                target_nodata = None
                if self.target_nodata_input.value.strip():
                    try:
                        target_nodata = float(self.target_nodata_input.value.strip())
                    except ValueError:
                        print(f"Warning: Invalid target nodata value '{self.target_nodata_input.value}'. Using auto-detection.")
                
                result = merge_tifs_general(
                    subdir,
                    process_nodata=True,
                    unify_nodata=self.unify_nodata_checkbox.value,
                    target_nodata=target_nodata
                )
            return (True, subdir, result)
        except Exception as e:
            return (False, subdir, str(e))
    
    def on_process_button_clicked(self, b):
        self.output.clear_output()
        with self.output:
            directory = self.directory_input.value
            if not directory or not os.path.isdir(directory):
                print(f"Error: {directory} is not a valid directory.")
                return
            
            print(f"Mode: {self.merge_mode.label}")
            print(f"Output files will be saved in: {directory}")
            print()
            
            subdirs = [os.path.join(directory, subdir) for subdir in os.listdir(directory) 
                       if os.path.isdir(os.path.join(directory, subdir))]
            
            if not subdirs:
                print("No subdirectories found to process.")
                return
            
            print(f"Found {len(subdirs)} subdirectories to process.")
            print()
            
            # Process subdirectories in parallel
            with concurrent.futures.ProcessPoolExecutor() as executor:
                results = executor.map(self.process_subdir, subdirs)
                
                # Collect results
                success_count = 0
                for success, subdir, result in results:
                    if success:
                        success_count += 1
                    else:
                        print(f"ERROR processing {subdir}: {result}")
            
            print()
            print(f"Processing complete: {success_count}/{len(subdirs)} successful.")
            
            if self.delete_checkbox.value:
                print()
                print("Deleting original subfolders...")
                for subdir in subdirs:
                    try:
                        shutil.rmtree(subdir)
                        print(f"Deleted: {subdir}")
                    except Exception as e:
                        print(f"Error deleting {subdir}: {e}")
                print("Original subfolders have been deleted.")
            else:
                print("Original subfolders have been kept.")
            
            print()
            print("Pre-processing completed.")

# Create and display the tool
tool = GeneralTilesMosaicingTool()

# MANUAL RUN

Use the cells below if you prefer to run the merger programmatically.

In [ ]:
import os
import sys
import concurrent.futures
import shutil

# Add utility path
utility_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'utility'))
if utility_path not in sys.path:
    sys.path.insert(0, utility_path)

from merge_utils_general import merge_tifs_general, merge_tifs_fathom

# CONFIGURATION
# Specify the directory containing the tile subfolders
directory = 'C:/YourWorkDirectory/data/tiles/'

# Choose merge function:
# - merge_tifs_general: Preserves negative values, auto-detects nodata
# - merge_tifs_fathom: Converts negatives to 0, sets nodata=0
merge_function = merge_tifs_general

# Optional: specify target nodata (None for auto-detection)
target_nodata = None

# Optional: unify multiple nodata values
unify_nodata = True

# Delete original subfolders after processing?
delete_originals = False

In [ ]:
print(f"Output files will be saved in: {directory}")
print()

# Get subdirectories
subdirs = [os.path.join(directory, subdir) for subdir in os.listdir(directory) 
           if os.path.isdir(os.path.join(directory, subdir))]

if not subdirs:
    print("No subdirectories found to process.")
else:
    print(f"Found {len(subdirs)} subdirectories to process.")
    print()
    
    # Process subdirectories
    def process_subdir(subdir):
        if merge_function == merge_tifs_fathom:
            return merge_tifs_fathom(subdir)
        else:
            return merge_tifs_general(subdir, process_nodata=True, 
                                     unify_nodata=unify_nodata, 
                                     target_nodata=target_nodata)
    
    # Process in parallel
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(process_subdir, subdirs))
    
    print()
    print("Processing complete.")
    
    # Delete originals if requested
    if delete_originals:
        user_input = input("Do you want to delete the original subfolders? (Y/N): ").strip().lower()
        if user_input == 'y':
            for subdir in subdirs:
                try:
                    shutil.rmtree(subdir)
                    print(f"Deleted: {subdir}")
                except Exception as e:
                    print(f"Error deleting {subdir}: {e}")
            print("Original subfolders have been deleted.")
        else:
            print("Original subfolders have been kept.")
    else:
        print("Original subfolders have been kept.")
    
    print()
    print("Pre-processing completed.")